In [2]:
from typing import TypedDict
from sqlalchemy import create_engine, text
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

In [3]:
llm = ChatOllama(model="llama3")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [4]:
engine = create_engine("sqlite:///example.db")

with engine.begin() as conn:  # ✅ commit fix
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER
    )
    """))

    conn.execute(text("DELETE FROM users"))

    conn.execute(text("""
    INSERT INTO users (name, age) VALUES
    ('Ravi', 30),
    ('Anil', 25),
    ('Sita', 28)
    """))

In [5]:
docs = [
    Document(page_content="RAG stands for Retrieval-Augmented Generation."),
    Document(page_content="LangGraph is used for multi-agent workflows."),
    Document(page_content="Kubernetes is for container orchestration.")
]

vectorstore = FAISS.from_documents(docs, embeddings)

In [6]:
def web_agent(state):
    print("→ Web Agent")

    prompt = f"Give latest info about: {state['query']}"
    res = llm.invoke(prompt)

    return {"result": res.content}

In [7]:
def sql_agent(state):
    print("→ SQL Agent")

    prompt = f"""
Convert to SQLite SQL.
ONLY SQL. NO explanation.

Query: {state['query']}
"""

    raw = llm.invoke(prompt).content

    # extract SQL safely
    sql = raw.strip().split("\n")[-1]

    if not sql.lower().startswith("select"):
        return {"result": "Invalid SQL generated"}

    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql))
            rows = result.fetchall()
        return {"result": str(rows)}
    except Exception as e:
        return {"result": f"SQL Error: {str(e)}"}

In [8]:
def rag_agent(state):
    print("→ RAG Agent")

    docs = vectorstore.similarity_search(state["query"], k=2)
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
Answer ONLY from context.
If not found, say "I don't know".

Context:
{context}

Question: {state['query']}
"""

    res = llm.invoke(prompt)
    return {"result": res.content}

In [9]:
def router(state):
    query = state["query"].lower()

    print("Routing Query:", query)

    # ✅ RULE BASED FIRST (FAST + RELIABLE)
    if any(x in query for x in ["user", "table", "database", "sql"]):
        route = "sql"

    elif any(x in query for x in ["news", "latest", "today"]):
        route = "web"

    elif any(x in query for x in ["what is", "explain", "define"]):
        route = "rag"

    else:
        # 🔥 LLM fallback
        prompt = f"""
Classify into one word: web, sql, rag

Query: {query}
"""
        raw = llm.invoke(prompt).content.lower()

        if "sql" in raw:
            route = "sql"
        elif "web" in raw:
            route = "web"
        else:
            route = "rag"

    print("FINAL ROUTE:", route)

    return {"route": route}

In [10]:
class AgentState(TypedDict):
    query: str
    route: str
    result: str

graph = StateGraph(AgentState)

graph.add_node("router", router)
graph.add_node("web", web_agent)
graph.add_node("sql", sql_agent)
graph.add_node("rag", rag_agent)

graph.add_conditional_edges(
    "router",
    lambda x: x["route"],
    {
        "web": "web",
        "sql": "sql",
        "rag": "rag"
    }
)

graph.add_edge("web", END)
graph.add_edge("sql", END)
graph.add_edge("rag", END)

graph.set_entry_point("router")

app_graph = graph.compile()

In [11]:
def run_query(q):
    result = app_graph.invoke({"query": q})
    print("\nFinal Result:\n", result["result"])

In [12]:
run_query("latest AI news")

Routing Query: latest ai news
FINAL ROUTE: web
→ Web Agent



Final Result:
 Here are some of the latest AI news and developments:

**1. Google's Latest AI Breakthrough:**

Google has achieved a major milestone in natural language processing (NLP) with its new AI model, called "Bloom." Bloom is capable of generating text that is virtually indistinguishable from human-written text, marking a significant step forward in the development of conversational AI.

**2. AI-Powered Medical Imaging:**

Researchers at Stanford University have developed an AI-powered system that can detect breast cancer more accurately than human radiologists. The system uses deep learning algorithms to analyze mammography images and identify tumors with high precision.

**3. AI-Generated Synthetic Data:**

A new AI algorithm has been developed to generate synthetic data that is indistinguishable from real-world data. This technology could revolutionize industries such as healthcare, finance, and marketing by providing a more cost-effective and privacy-preserving way of trai

In [13]:
run_query("show all users")

Routing Query: show all users
FINAL ROUTE: sql
→ SQL Agent



Final Result:
 [(1, 'Ravi', 30), (2, 'Anil', 25), (3, 'Sita', 28)]


In [14]:
run_query("what is RAG?")

Routing Query: what is rag?
FINAL ROUTE: rag
→ RAG Agent



Final Result:
 Retrieval-Augmented Generation.


In [15]:
run_query("what is Agentic AI?")

Routing Query: what is agentic ai?
FINAL ROUTE: rag
→ RAG Agent



Final Result:
 I don't know.


In [16]:
run_query("Get user data and explain trends from latest AI news")

Routing Query: get user data and explain trends from latest ai news
FINAL ROUTE: sql
→ SQL Agent

Final Result:
 Invalid SQL generated
